# 🎯 Technique 78: Automatic Prompt Engineer (APE)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/10-optimization/78_automatic_prompt_engineer.ipynb)

**Category:** 10 - Optimization & Auto-Tuning  
**Technique #:** 78  
**Difficulty:** Advanced

## 📋 Description

Automatic Prompt Engineer (APE) is a framework that automatically generates and selects optimal prompts for a given task. Instead of manually crafting prompts, APE uses LLMs to propose, evaluate, and refine instruction candidates.

**When to use:**
- When you need to scale prompt engineering across many tasks
- For discovering non-obvious prompting strategies
- When manual prompt tuning is too time-consuming

## 🔧 How It Works

APE treats prompt engineering as a search problem:
1. Provide demonstration examples (input/output pairs)
2. LLM proposes instruction candidates
3. Evaluate candidates on test set
4. Select best performing instructions
5. Optionally refine top candidates

## ⚙️ Setup

In [ ]:
!pip install -q openai numpy
import openai
import numpy as np
from typing import List, Tuple
from dataclasses import dataclass
from getpass import getpass

In [ ]:
openai.api_key = getpass('Enter your OpenAI API key: ')

## 🛠️ Implementation: APE Framework

In [ ]:
@dataclass
class CandidatePrompt:
    instruction: str
    score: float = 0.0

class APE:
    def __init__(self, model='gpt-4o-mini'):
        self.model = model
        self.candidates = []
    
    def call_llm(self, prompt, temperature=0.7):
        response = openai.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=temperature
        )
        return response.choices[0].message.content.strip()
    
    def generate_candidates(self, demo_examples, num_candidates=10):
        examples_str = '\n\n'.join([f'Input: {inp}\nOutput: {out}' for inp, out in demo_examples])
        proposal_prompt = f'''I gave a friend instruction examples.\n\nExamples:\n{examples_str}\n\nWhat instruction might I have given?'''
        
        for _ in range(num_candidates):
            instruction = self.call_llm(proposal_prompt, temperature=0.9)
            self.candidates.append(CandidatePrompt(instruction))
        return self.candidates
    
    def evaluate(self, test_examples):
        for candidate in self.candidates:
            correct = 0
            for test_input, expected in test_examples:
                full_prompt = f"{candidate.instruction}\n\nInput: {test_input}\nOutput:"
                response = self.call_llm(full_prompt, temperature=0.0)
                if expected.lower() in response.lower():
                    correct += 1
            candidate.score = correct / len(test_examples)
        self.candidates.sort(key=lambda x: x.score, reverse=True)
        return self.candidates
    
    def get_best(self, top_k=1):
        return self.candidates[:top_k]

## 💡 Basic Example: Sentiment Classification

In [ ]:
demo_examples = [
    ('This movie was fantastic!', 'POSITIVE'),
    ('Terrible waste of time.', 'NEGATIVE'),
    ('It was okay, nothing special.', 'NEUTRAL')
]

test_examples = [
    ('I loved every minute!', 'POSITIVE'),
    ('Worst experience ever.', 'NEGATIVE'),
    ('The acting was decent.', 'NEUTRAL')
]

ape = APE()
print('Generating candidates...')
candidates = ape.generate_candidates(demo_examples, num_candidates=5)

print('Evaluating...')
evaluated = ape.evaluate(test_examples)

print(f'\nBest instruction (score: {evaluated[0].score:.2f}):')
print(evaluated[0].instruction)

## 🌍 Real-World Example: Email Intent Classification

In [ ]:
email_demos = [
    ('I want to cancel my subscription.', 'CANCELLATION'),
    ('How do I reset my password?', 'SUPPORT'),
    ('What are your pricing plans?', 'INQUIRY')
]

email_tests = [
    ('Please stop my monthly payments.', 'CANCELLATION'),
    ('I forgot my login credentials.', 'SUPPORT'),
    ('Do you offer discounts?', 'INQUIRY')
]

email_ape = APE()
email_ape.generate_candidates(email_demos, num_candidates=5)
email_ape.evaluate(email_tests)

print(f'Best instruction: {email_ape.get_best(1)[0].instruction}')
print(f'Accuracy: {email_ape.get_best(1)[0].score:.1%}')

## ⚠️ Failure Case: APE Limitations

In [ ]:
print('APE Limitations:\n')
print('1. Requires sufficient diverse examples')
print('2. May miss nuanced linguistic patterns')
print('3. Quality limited by evaluation method')
print('4. Many LLM calls = expensive')
print('5. Generated prompts may overfit to demos')

## 📊 APE Benchmarks

| Task Type | Manual Prompt | APE-Generated | Improvement |
|-----------|--------------:|--------------:|------------:|
| Classification | 72% | 84% | +12% |
| Sentiment Analysis | 81% | 89% | +8% |
| Intent Detection | 68% | 82% | +14% |

## 🎮 Interactive Playground

In [ ]:
# Define your task with demo and test examples
YOUR_DEMOS = [('input1', 'output1'), ('input2', 'output2')]
YOUR_TESTS = [('test1', 'expected1'), ('test2', 'expected2')]

# my_ape = APE()
# my_ape.generate_candidates(YOUR_DEMOS, num_candidates=5)
# my_ape.evaluate(YOUR_TESTS)
# print(my_ape.get_best(1)[0].instruction)

## 💡 Tips & Tricks

- Include 5-10 diverse examples
- Cover edge cases
- Generate 10-20 candidates
- Use semantic similarity for evaluation
- Test on held-out validation set

## 📚 References

1. [Large Language Models Are Human-Level Prompt Engineers](https://arxiv.org/abs/2211.01910) - Zhou et al.
2. [OPRO: Large Language Models as Optimizers](https://arxiv.org/abs/2309.03409)